# k8 — Viết lại câu hỏi + thuật ngữ mở rộng (mirret/query_expand.py)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On.
**Add Input:** `mir2-code` (bản mới có `mirret/query_expand.py`), `mir2-data`, `mir2-chunks` (chỉ mục BM25 — cho ô (5)),
dataset có `qtrans/queries_{vi,en,zh}.parquet` (output k4 `mir2-runs` hoặc `mir2-qtrans`), dataset có file câu hỏi
`80queries.csv`; thêm output k8 phiên trước (`mir2-qx`) nếu chạy tiếp — lượt đã có trong cache không sinh lại.

Qwen2.5-14B (4-bit) sinh cho mỗi câu × {vi, en, zh} một JSON `{"rewrite", "terms"}` (greedy). vi = câu gốc, en/zh =
bản dịch t2 (cùng bản baseline). Gợi ý thuật ngữ từ `configs/med_glossary_vi.tsv`.
**Thử 5 câu** (mặc định): `LIMIT = 5` → 15 lượt, 1 GPU, ~10 phút kể cả nạp mô hình.
**Đủ 80 câu / 1.200 câu:** `LIMIT = 0` (+ `QIDS = ''` cho 1.200 câu) → 2 GPU song song; ô (3) in s/lượt.
**Chia việc với máy khác** (vd Colab, notebook `c8_qx_colab.ipynb`): `SHARDS = ['0/3', '1/3']` (mỗi GPU một phần), máy kia
chạy `2/3`; mỗi phần ghi `records-<i>of<N>.jsonl` riêng, `build` gộp mọi file.
**Đầu ra:** `qexp/rw3-qwen25_14b/records*.jsonl` (cache theo phiên bản prompt, có đầu ra thô), `qexp/rw/`, `qexp/rwt/` → dataset **`mir2-qx`**.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k8-v4 (15/10/2026)'
MODEL = 'qwen25_14b'
QIDS = '80queries.csv'               # '' = cả 1.200 câu
LIMIT = 5                            # >0: chỉ N câu đầu (thử); 0 = mọi câu trong QIDS
SHARDS = []                          # vd ['0/3', '1/3']: mỗi GPU một phần (phần 2/3 chạy ở máy khác)
BATCH, QUANT = 8, '4bit'
QID_EXPLAIN = None                   # câu in token + trọng số BM25 (None = câu nhỏ nhất)
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
def _ver(h):                                                  # output notebook cũ (vd mir2-chunks/code) cũng có mirret/
    v = os.path.join(os.path.dirname(os.path.dirname(h)), 'VERSION.txt')
    return open(v, encoding='utf-8').read().strip() if os.path.exists(v) else ''
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True),
              key=lambda h: (_ver(h), -len(h)), reverse=True)   # bản mã mới nhất theo VERSION.txt (ngày giờ)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('qtrans',))

In [ ]:
# (2) Đầu vào: file câu hỏi, chỉ mục BM25, cache phiên trước; bản dịch đang dùng
if QIDS and not os.path.exists(QIDS) and not QIDS.replace(',', '').isdigit():
    nrow = lambda h: sum(1 for _ in open(h, encoding='utf-8-sig')) - 1
    hits = sorted(glob.glob(f'{KAGGLE}/input/**/{QIDS}', recursive=True), key=lambda h: (-nrow(h), len(h)))
    assert hits, f'Không thấy {QIDS} trong /kaggle/input (Add Input dataset chứa file này)'
    print('các file', QIDS, 'trong input:', [(h, nrow(h)) for h in hits])
    QIDS = hits[0]
QARG = f' --qids "{QIDS}"' if QIDS else ''
qt = locate('qtrans/queries_vi.parquet') / 'qtrans'
print('câu hỏi:', QIDS or 'cả 1.200 câu', '| bản dịch:', qt,
      open(qt / 'queries_meta.json', encoding='utf-8').read() if (qt / 'queries_meta.json').exists() else '')
bm = locate('index/bm25', required=False)
if bm is not None:
    K.set({'bm25.index_dir': str(bm / 'index' / 'bm25')})
else:
    print('[chú ý] không thấy index/bm25 (mir2-chunks) → ô (5) sẽ báo thiếu chỉ mục')
for d in locate_all('qexp'):                          # cache phiên trước → chỉ sinh phần còn thiếu
    for f in glob.glob(f'{d}/qexp/rw*-*/records*.jsonl'):
        dst = f'{K.work}/qexp/' + os.path.relpath(f, f'{d}/qexp')
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst); print('chép', f)
sh('python -m pip install -q -U bitsandbytes accelerate')

In [ ]:
# (3) Sinh rewrite + thuật ngữ (Qwen2.5-14B 4-bit). SHARDS: mỗi GPU một phần; thử ít câu: 1 GPU; còn lại: mỗi GPU một nửa
lim = f' --limit {LIMIT}' if LIMIT else ''
gen = f'python -m mirret.query_expand generate --config {K.cfg} --model {MODEL} --batch {BATCH} --quant {QUANT}{QARG}{lim}'
if SHARDS:
    par([f'CUDA_VISIBLE_DEVICES={g} {gen} --shard {s}' for g, s in enumerate(SHARDS)], 'qx_generate')
elif LIMIT and LIMIT <= 40:
    sh(f'CUDA_VISIBLE_DEVICES=0 {gen}', 'qx_generate')
else:
    par([f'CUDA_VISIBLE_DEVICES={g} {gen} --shard {g}/2' for g in range(2)], 'qx_generate')

In [ ]:
# (4) Dựng câu hỏi biến thể (qexp/rw, qexp/rwt) + in đầu ra 3 ngôn ngữ, số lượt lỗi JSON
sh(f'python -m mirret.query_expand build --config {K.cfg} --model {MODEL}')
sh(f'python -m mirret.query_expand show --config {K.cfg} --model {MODEL} --limit {LIMIT or 5}')

In [ ]:
# (5) Một câu: token + trọng số BM25 trước / sau mở rộng (w_base, w_rewrite, w_terms, max_df_ratio trong qexpand)
qa = f' --qid {QID_EXPLAIN}' if QID_EXPLAIN is not None else ''
sh(f'python -m mirret.query_expand explain --config {K.cfg}{qa}', check=False)
report('k8')
print('→ Save Version, tạo dataset mir2-qx từ output (qexp/)')